# KV cache: cómo funciona por dentro y qué cuesta

Carlos Alberto León Gil — [github.com/CarlosALeon](https://github.com/CarlosALeon)

## Por qué esto importa (aunque no programes)

Cada vez que le escribes a un asistente de inteligencia artificial y este responde,
por detrás alguien paga una factura de cómputo, y tú esperas unos segundos. Esos dos
costos —el dinero y la espera— dependen en buena parte de un detalle técnico que casi
nadie menciona: cómo el modelo recuerda lo que ya escribió.

Un modelo de lenguaje escribe palabra por palabra, y en cada palabra vuelve a leer
todo lo anterior. Es como si un secretario, para escribir la línea diez de una carta,
releyera desde la línea uno cada vez. Funciona, pero desperdicia tiempo. El KV cache
es la libreta donde el modelo anota lo que ya procesó para no repetir el trabajo.

Lo que se puede y lo que no, en una frase:

- **Sí** hace que las respuestas salgan mucho más rápido y más baratas.
- **Sí** es lo que permite que un asistente maneje conversaciones y documentos largos.
- **No** es gratis: esa libreta ocupa memoria, y mientras más largo el texto, más
  memoria consume, hasta volverse el verdadero límite.
- **No** cambia lo que el modelo responde: solo cambia qué tan rápido lo hace.

Este cuaderno reconstruye ese mecanismo desde cero, lo mide, y en cada paso explica
qué significa el número que aparece. Está basado en el material de Sebastian Raschka
sobre el tema; el código y las notas son propios.

## De dónde viene el desperdicio

La atención transforma cada token en tres vectores: query (Q), key (K) y value (V).
La cuenta que hace el modelo es, en esencia, `softmax(Q·Kᵀ/√d)·V`.

Hay un hecho que se aprovecha todo el tiempo pero rara vez se dice en voz alta: el K
y el V de un token dependen solo de ese token y de unos pesos que ya están fijos.
`k = x·W_k`, `v = x·W_v`. El token que va en la posición 1 produce el mismo par (k, v)
sin importar cuántas palabras vengan detrás. No cambian nunca. Así que recalcularlos
en cada paso es rehacer una cuenta cuyo resultado ya teníamos.

El query es distinto: en cada paso solo interesa el del token nuevo, porque es el que
pregunta "¿a qué le presto atención?". Por eso el query no se guarda; los que se
guardan son K y V. De ahí el nombre.

In [ ]:
import time
import torch
import torch.nn as nn

torch.manual_seed(123)
print("PyTorch:", torch.__version__)
print("Vamos a comparar dos formas de generar texto: recalculando todo en cada")
print("paso, y guardando K/V para reutilizarlos. La segunda es el KV cache.")

## La atención, con y sin memoria

El único bloque que cambia entre las dos versiones es qué hacemos con K y V. Con
cache, en vez de recalcularlos, los vamos acumulando. Lo demás es la misma atención
de siempre.

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)

    def reset_cache(self):
        # Entre dos textos distintos hay que borrar la memoria. Si no, el modelo
        # termina prestando atencion a palabras de un texto anterior.
        self.cache_k, self.cache_v = None, None

    def forward(self, x, use_cache=False):
        q     = self.W_query(x)
        k_new = self.W_key(x)
        v_new = self.W_value(x)

        if use_cache:
            # Solo calculamos K/V del token nuevo y los pegamos a lo que ya habia.
            if self.cache_k is None:
                self.cache_k, self.cache_v = k_new, v_new
            else:
                self.cache_k = torch.cat([self.cache_k, k_new], dim=1)
                self.cache_v = torch.cat([self.cache_v, v_new], dim=1)
            k, v = self.cache_k, self.cache_v
        else:
            # Sin memoria: se recalcula K/V de toda la secuencia, cada vez.
            k, v = k_new, v_new

        scores = q @ k.transpose(-2, -1) / (self.d_out ** 0.5)
        Tq, Tk = q.shape[1], k.shape[1]
        offset = Tk - Tq
        mask = torch.triu(
            torch.ones(Tq, Tk, device=x.device, dtype=torch.bool),
            diagonal=1 + offset,
        )
        scores = scores.masked_fill(mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        return attn @ v

print("Atencion causal definida: la version con use_cache=True guarda K/V;")
print("la version con use_cache=False los recalcula desde cero en cada paso.")

Ese `offset` de la máscara es el tipo de detalle que parece menor y no lo es.
Sin cache, entran tantos queries como keys y la máscara triangular normal alcanza.
Con cache, en cada paso entra **un solo** query contra **toda** la historia de keys.
Ese query está en la última posición, así que tiene derecho a mirar todo lo anterior;
`offset = Tk - Tq` corre la diagonal para que la máscara lo permita. Equivocarse aquí
no rompe el programa: simplemente el texto empieza a diferir del correcto, que es la
peor clase de error porque pasa desapercibido.

## El modelo mínimo

Un GPT de juguete: embeddings, una capa de atención, y una cabeza que predice el
siguiente token. El único añadido frente a un modelo normal es `current_pos`, que
lleva la cuenta de cuántos tokens ya pasaron. Hace falta porque la posición de una
palabra tiene que seguir la numeración real; si cada token nuevo se numerara desde
cero, el modelo creería que todos ocupan el mismo lugar y se confundiría.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, ctx_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(ctx_len, d_model)
        self.att = CausalAttention(d_model, d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.ctx_len = ctx_len
        self.current_pos = 0

    def reset_cache(self):
        self.att.reset_cache()
        self.current_pos = 0

    def forward(self, idx, use_cache=False):
        _, T = idx.shape
        if use_cache:
            pos = torch.arange(self.current_pos, self.current_pos + T, device=idx.device)
            self.current_pos += T
        else:
            pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos).unsqueeze(0)
        x = self.att(x, use_cache=use_cache)
        return self.head(x)

print("Modelo definido. current_pos es el contador de posicion que mantiene")
print("alineadas las palabras nuevas con las que ya estan en la memoria.")

## Generar: dos fases

Con cache, generar tiene dos momentos. Primero el *prefill*: se procesa el prompt
entero de una vez y queda guardado en la memoria. Después el *decode*: por cada
palabra nueva se alimenta **solo esa palabra**, no todo el texto. Ahí está el ahorro
completo. Sin cache no hay fases: cada paso reprocesa todo desde el principio.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, use_cache):
    model.eval()
    if use_cache:
        model.reset_cache()
        logits = model(idx, use_cache=True)               # prefill: el prompt entero
        for _ in range(max_new_tokens):
            next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_idx], dim=1)
            logits = model(next_idx, use_cache=True)       # decode: solo el token nuevo
    else:
        for _ in range(max_new_tokens):
            logits = model(idx[:, -model.ctx_len:], use_cache=False)  # todo, cada vez
            next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_idx], dim=1)
    return idx

print("Funcion de generacion lista. Con cache: prefill una vez, luego decode")
print("alimentando un solo token por paso. Sin cache: reprocesa todo cada vez.")

## Primero, que dé lo mismo

Antes de hablar de velocidad hay que asegurar una cosa: la versión con cache tiene
que producir exactamente el mismo texto que la versión sin cache. Si cambia el
resultado, no optimizamos nada, introdujimos un error. Este chequeo es lo que separa
una optimización real de un bug disfrazado.

In [ ]:
vocab_size, d_model, ctx_len = 100, 64, 512
model = MiniGPT(vocab_size, d_model, ctx_len)
prompt = torch.randint(0, vocab_size, (1, 4))
N = 200

out_no  = generate(model, prompt.clone(), N, use_cache=False)
out_yes = generate(model, prompt.clone(), N, use_cache=True)

iguales = torch.equal(out_no, out_yes)
print("Mismo texto con y sin cache:", iguales)
if iguales:
    print("Significa que el cache es transparente: cambia el 'cómo', no el 'qué'.")
    print("La salida es idéntica token a token, así que la optimización es válida.")
else:
    print("Divergen: hay un error de indexación (típicamente en la máscara o la posición).")
assert iguales

## Ahora sí, la velocidad

El modelo no está entrenado, así que el texto que produce es ruido. No importa para
lo que medimos: el cache no decide *qué* palabra sale, solo *cuánto tarda* en salir.
Generamos 200 tokens de las dos formas y comparamos.

In [ ]:
def medir(use_cache, reps=3):
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        generate(model, prompt.clone(), N, use_cache=use_cache)
        best = min(best, time.perf_counter() - t0)
    return best

t_no  = medir(False)
t_yes = medir(True)
print(f"Recalculando todo : {t_no:.3f} s")
print(f"Con KV cache      : {t_yes:.3f} s")
print(f"Diferencia        : {t_no / t_yes:.2f}x más rápido")
print()
print(f"En la práctica esto es {t_no/t_yes:.1f} veces menos tiempo para el mismo texto.")
print("El ahorro viene de no rehacer los K/V de las palabras que ya estaban.")

## Por qué el ahorro crece con el texto

Aquí aparece la parte teórica que se cita en casi todos los artículos. Sin cache,
para escribir el token número *t* hay que comparar contra los *t* anteriores, y eso
se hace en cada uno de los *t* pasos: el trabajo total crece con el cuadrado de la
longitud, O(n²). Con cache, cada K/V se calcula una sola vez, así que el trabajo crece
de forma lineal, O(n). La diferencia entre una curva cuadrática y una lineal no se
nota en textos cortos, pero se dispara en los largos. La tabla lo muestra: a más
tokens, más grande la ventaja.

In [ ]:
def correr(uc, n):
    t0 = time.perf_counter()
    generate(model, prompt.clone(), n, uc)
    return time.perf_counter() - t0

print(f"{'tokens':>7} {'sin cache':>11} {'con cache':>11} {'ventaja':>9}")
for n in [50, 100, 200, 400]:
    a = min(correr(False, n) for _ in range(2))
    b = min(correr(True, n)  for _ in range(2))
    print(f"{n:7d} {a:10.3f}s {b:10.3f}s {a/b:8.2f}x")
print()
print("La columna 'ventaja' sube al bajar por la tabla. Eso es, en vivo, el paso")
print("de O(n²) a O(n): mientras más largo el texto, más trabajo repetido se evita.")

## Qué se paga por esto

Nada es gratis. El cache guarda un par (K, V) por cada token, así que la memoria crece
de forma lineal con la longitud del texto. En un modelo grande con contexto largo, esa
memoria puede terminar pesando más que el propio modelo. Vale la pena ver el tamaño
real con las cifras de un modelo tipo Llama, para que el problema deje de ser
abstracto.

In [ ]:
# Cuenta de servilleta para un modelo tipo Llama 3 8B.
capas       = 32
kv_heads    = 8          # con Grouped-Query Attention (GQA); con atención normal serían 32
head_dim    = 128
bytes_num   = 2          # fp16
def kv_gb(tokens):
    # x2 = K y V; por capa, por cabeza, por dimensión, por token
    return 2 * capas * kv_heads * head_dim * tokens * bytes_num / (1024**3)

print("Memoria del KV cache según la longitud del contexto (Llama 3 8B, GQA):")
for ctx in [1_000, 8_000, 32_000, 128_000]:
    print(f"  {ctx:>7,} tokens  ->  {kv_gb(ctx):6.2f} GB")
print()
print("A 128k tokens el cache ronda los 15 GB, comparable al peso del modelo.")
print("Por eso 'cuánto contexto soporto' no es pregunta del modelo, sino de cuánta")
print("memoria queda para el cache. Y por eso GQA existe: si en vez de 8 cabezas")
print("de K/V usáramos las 32 completas, esta tabla se multiplicaría por cuatro.")

## Lo que la investigación hizo con este problema

El cache básico no es tema de artículo: es infraestructura conocida. Lo que sí llena
arXiv es el reto que abre —la memoria— y cómo domarla. Conviene leer estos trabajos
con la tabla de GB anterior en la cabeza, porque todos atacan esa misma cifra. Estos
son diez del último año, con lo que cada uno encontró:

1. **Learning to Evict from Key-Value Cache** (arXiv:2602.10238). En vez de reglas
   fijas para decidir qué tokens conservar, entrenan un criterio que aprende cuáles
   se pueden botar; reportan mantener la calidad descartando buena parte del cache.
2. **Semantic-Retrieval-Guided KV-Cache Compression** (arXiv:2606.24467). Encuentran
   que decidir qué comprimir según la relevancia semántica del contenido conserva más
   precisión en contexto largo que las reglas ciegas por posición.
3. **PolyKV: Shared Asymmetrically-Compressed KV Cache Pool** (arXiv:2604.24971). Su
   resultado concreto: con 15 agentes compartiendo un contexto de 4k tokens, bajan el
   cache de 19.8 GB a 0.45 GB (una reducción del 97.7%) perdiendo solo 0.57% de
   precisión.
4. **Compressed and Composable KV Cache Reuse** (arXiv:2607.17715, KDD '26). Muestran
   que se puede reutilizar el cache entre distintas peticiones que comparten el mismo
   comienzo, evitando recalcular el prefijo común.
5. **KV Cache Optimization Strategies for Scalable and Efficient LLM Inference**
   (arXiv:2603.20397). Es un panorama: clasifican las técnicas existentes y confirman
   que el cache es la optimización base sobre la que se construye todo lo demás.
6. **KV Cache Transform Coding for Compact Storage** (arXiv:2511.01815). Aplican
   codificación por transformada (la misma idea detrás de comprimir imágenes) para
   guardar el cache ocupando menos y poder reusarlo entre turnos de conversación.
7. **Inference-Time Hyper-Scaling with KV Cache Compression** (arXiv:2506.05345).
   Su hallazgo: al comprimir el cache queda presupuesto para generar más tokens, y eso
   sube la precisión final si la compresión no degrada la información.
8. **Synthesizing Recurrence with KV Cache Compression** (arXiv:2402.09398). Observan
   que la poda pura falla cuando la tarea exige recordar casi todo, y proponen mezclar
   recurrencia con compresión para no perder esa memoria.
9. **An Efficient KV Cache Layer for Enterprise-Scale LLM Inference**
   (arXiv:2510.09665). Sacan el cache de la GPU para reutilizarlo entre consultas y
   entre motores de inferencia distintos, atacando el costo de recomputar prefijos.
10. **KV Cache Compression for Inference Efficiency in LLMs: A Review**
    (arXiv:2508.06297). Revisión que ordena las familias de compresión y sus
    compromisos de precisión frente a memoria.

Los cimientos que aparecen citados una y otra vez: **MQA** (Shazeer, 2019,
arXiv:1911.02150) y **GQA** (Ainslie et al., 2023, arXiv:2305.13245), que comparten
K/V entre cabezas —la técnica que usamos en la cuenta de memoria de arriba, y la razón
de que fueran 8 cabezas y no 32—; **H2O** (arXiv:2306.14048), que encontró que
conservar solo los tokens más atendidos (los "heavy hitters") basta para casi la misma
calidad; y **PagedAttention / vLLM** (Kwon et al., 2023), que administra el cache como
memoria virtual paginada y con eso redujo el desperdicio por fragmentación, subiendo el
número de peticiones que caben a la vez.

## Cuándo sí, cuándo no

Sirve cuando se genera token por token y el texto va creciendo: chat, generación larga,
cualquier cosa que se sirva con latencia por token. No sirve en entrenamiento, donde
toda la secuencia entra de una vez y no hay nada que reutilizar, ni en un único pase
sobre un texto fijo (clasificar, sacar embeddings). Y en contextos muy largos deja de
alcanzar por memoria, salvo que además se comprima, se descarte o se comparta —que es
exactamente lo que hacen los diez trabajos de arriba.

La conclusión honesta: el mecanismo se entiende en diez minutos, pero saber cuándo
aplica, qué cuesta y dónde se rompe es lo que separa copiar un `torch.cat` de entender
por qué la inferencia moderna se organiza alrededor de esta memoria.

---

Basado en el material de Sebastian Raschka. Referencias vía arXiv. Código y notas:
Carlos A. León — [github.com/CarlosALeon](https://github.com/CarlosALeon).